In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import requests
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from plotly_resampler import FigureResampler
import plotly.graph_objects as go

In [ ]:
VIRTUOSO_URL = "http://localhost:8890/sparql"
GRAPH_URI    = "http://example.com/Gent-Terneuzen"
USERNAME     = "dba"
PASSWORD     = "dba"
AUTH         = (USERNAME, PASSWORD)

In [ ]:
params  = {'graph': GRAPH_URI}
headers = {'Accept': 'text/turtle'}

# Identifying unique sensors

In [ ]:
sensor_set = set()

sensor_query = f"""
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    SELECT DISTINCT ?sensor
    WHERE {{
        GRAPH <{GRAPH_URI}> {{
            ?obs a sosa:Observation ;
                 sosa:madeBySensor ?sensor .
        }}
    }}
    """
res = requests.get(VIRTUOSO_URL, params={'query': sensor_query, 'format': 'application/sparql-results+json'})
if res.status_code != 200:
    print(f"Error: {res.status_code}")
    print("Response:", res.text)
else:
    print("Unique sensors identified successfully!")

data     = res.json()
bindings = data['results']['bindings']
for row in bindings:
    sensor_set.add(row['sensor']['value'])

print(f"Added {len(sensor_set)} unique sensors to the set.")
print("Sensors:", sensor_set)

# Reframe the data

In [ ]:
final_df = pd.DataFrame()
print("Fetching and pivoting sensor data...")

for sensor_uri in sensor_set:
    column_name = sensor_uri.split('/')[-1]
    query = f"""
        PREFIX sosa: <http://www.w3.org/ns/sosa/>
        PREFIX ex: <http://example.com/attributes/>
        SELECT ?time ?value ?unixtime
        WHERE {{
            GRAPH <{GRAPH_URI}> {{
                ?obs a sosa:Observation ;
                    sosa:resultTime ?time ;
                    sosa:hasSimpleResult ?value ;
                    ex:unixTimestamp ?unixtime ;
                    sosa:madeBySensor <{sensor_uri}> .
            }}
        }}
    """
    res = requests.get(VIRTUOSO_URL, params={'query': query, 'format': 'application/sparql-results+json'})
    if res.status_code == 200:
        bindings = res.json()['results']['bindings']
        temp_data = [
            {'time': row['time']['value'], column_name: float(row['value']['value']),
             'unixtime': int(row['unixtime']['value'])}
            for row in bindings
        ]
        temp_df = pd.DataFrame(temp_data)
        if not temp_df.empty:
            temp_df['time'] = pd.to_datetime(temp_df['time'])
            if final_df.empty:
                final_df = temp_df
            else:
                final_df = pd.merge(final_df, temp_df, on=['time', 'unixtime'], how='outer')
            print(f"Added column for sensor: {column_name}")

final_df = final_df.sort_values('time').set_index('time')
print("Finished!")
print(final_df.head())

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(12, 10), sharex=True)

sensors = ['289435042', '289423042', '289429042', '289441042']
colors  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, sensor in enumerate(sensors):
    ax = axes[i]
    ax.plot(final_df.index, final_df[sensor], label=f"Sensor {sensor}", color=colors[i])
    ax.set_ylabel("Value")
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle("Sensor Data Analysis", fontsize=16)
plt.xlabel("Time")
plt.xticks(rotation=45)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Pre-processing

In [ ]:
df_clean = final_df.copy()
df_clean = df_clean.ffill().bfill()

feature_sensors = ['289435042', '289423042', '289429042', '289441042']
target_sensor   = '289441042'
target_col_idx  = feature_sensors.index(target_sensor)   # = 3
n_features      = len(feature_sensors)                   # = 4

data_arr = df_clean[feature_sensors].values               # (n_samples, 4)

# Split FIRST, then fit scaler on training data only
split_index  = int(len(data_arr) * 0.8)
train_data   = data_arr[:split_index]
test_data    = data_arr[split_index:]

scaler        = MinMaxScaler(feature_range=(0, 1))
train_scaled  = scaler.fit_transform(train_data)
test_scaled   = scaler.transform(test_data)

print(f"Train samples: {len(train_scaled):,}  |  Test samples: {len(test_scaled):,}")

# Sliding-window sequences

Unlike the original notebook, sequences are kept as **3-D tensors `(samples, time_steps, n_features)`** so the LSTM encoder can learn temporal ordering inside each window.

In [ ]:
time_steps = 672   # 1 week of 15-min readings
n_future   = 96    # 24 hours ahead

def create_sequences(dataset, time_steps, n_future, target_col):
    """Return X of shape (samples, time_steps, n_features)
       and y of shape (samples, n_future)."""
    X, y = [], []
    for i in range(len(dataset) - time_steps - n_future + 1):
        X.append(dataset[i : i + time_steps, :])                               # (time_steps, n_features)
        y.append(dataset[i + time_steps : i + time_steps + n_future, target_col])  # (n_future,)
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, time_steps, n_future, target_col_idx)
X_test,  y_test  = create_sequences(test_scaled,  time_steps, n_future, target_col_idx)

print(f"X_train: {X_train.shape}  →  (samples, time_steps, n_features)")
print(f"y_train: {y_train.shape}  →  (samples, n_future)")

# Stage 1 — LSTM denoising autoencoder (pre-training)

**What changed from the original:**
- Encoder is now two stacked LSTM layers; the final hidden state is the bottleneck (32 dims instead of 8).
- Decoder uses `RepeatVector` + two LSTM layers + `TimeDistributed(Dense)` to reconstruct all 4 channels at every time step.
- Gaussian noise is still injected on the input so the model learns robust representations.
- `EarlyStopping` (patience 10) and `ReduceLROnPlateau` replace the fixed 5-epoch runs.

In [ ]:
def build_lstm_autoencoder(time_steps, n_features, bottleneck=32,
                           dropout_rate=0.1, l2_reg=1e-4, noise_std=0.05):
    reg = regularizers.l2(l2_reg)

    # ── Encoder ──────────────────────────────────────────────────────────────
    enc_input = layers.Input(shape=(time_steps, n_features), name='enc_input')
    x = layers.GaussianNoise(noise_std)(enc_input)

    x = layers.LSTM(64, return_sequences=True,
                    kernel_regularizer=reg, name='lstm_enc_1')(x)
    x = layers.Dropout(dropout_rate)(x)

    z = layers.LSTM(bottleneck, return_sequences=False,
                    kernel_regularizer=reg, name='bottleneck')(x)  # (batch, bottleneck)

    # ── Decoder ──────────────────────────────────────────────────────────────
    x = layers.RepeatVector(time_steps)(z)                          # (batch, time_steps, bottleneck)

    x = layers.LSTM(bottleneck, return_sequences=True,
                    kernel_regularizer=reg, name='lstm_dec_1')(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.LSTM(64, return_sequences=True,
                    kernel_regularizer=reg, name='lstm_dec_2')(x)

    dec_output = layers.TimeDistributed(
        layers.Dense(n_features, activation='sigmoid'), name='reconstruction'
    )(x)                                                            # (batch, time_steps, n_features)

    autoencoder = Model(enc_input, dec_output, name='lstm_autoencoder')
    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='mse'
    )
    return autoencoder


autoencoder = build_lstm_autoencoder(
    time_steps=time_steps,
    n_features=n_features,
    bottleneck=32
)
autoencoder.summary()

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
early_stop_ae = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
)
reduce_lr_ae = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
)

# ── Pre-train on reconstruction (target = input) ──────────────────────────
history_ae = autoencoder.fit(
    X_train, X_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop_ae, reduce_lr_ae],
    verbose=1
)

# ── Plot pre-training loss ────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(history_ae.history['loss'],     label='Train loss')
plt.plot(history_ae.history['val_loss'], label='Val loss')
plt.title('Stage 1 — Autoencoder reconstruction loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Stage 2 — Attach forecasting head and fine-tune

The encoder is extracted from the autoencoder, frozen, and a `Dense(n_future)` head is attached to the 32-dim bottleneck.  
Training happens in two phases:
1. **Frozen encoder** — only the forecast head trains (fast convergence, no risk of destroying pre-trained weights).
2. **Unfrozen encoder** — full end-to-end fine-tuning at a much lower learning rate (1e-5) plus `ReduceLROnPlateau`.

In [ ]:
# ── Extract encoder ───────────────────────────────────────────────────────
encoder = Model(
    inputs  = autoencoder.input,
    outputs = autoencoder.get_layer('bottleneck').output,
    name    = 'lstm_encoder'
)
encoder.summary()

# ── Freeze encoder weights ────────────────────────────────────────────────
for layer in encoder.layers:
    layer.trainable = False

# ── Build forecaster ──────────────────────────────────────────────────────
fc_input  = layers.Input(shape=(time_steps, n_features), name='fc_input')
z         = encoder(fc_input)                   # (batch, 32)
fc_output = layers.Dense(n_future, name='forecast')(z)  # (batch, 96)

forecaster = Model(fc_input, fc_output, name='forecaster')
forecaster.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='mse'
)
forecaster.summary()

In [ ]:
# ── Phase 1: train head only ──────────────────────────────────────────────
early_stop_p1 = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
)
reduce_lr_p1 = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
)

history_p1 = forecaster.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop_p1, reduce_lr_p1],
    verbose=1
)

plt.figure(figsize=(10, 4))
plt.plot(history_p1.history['loss'],     label='Train loss')
plt.plot(history_p1.history['val_loss'], label='Val loss')
plt.title('Stage 2 Phase 1 — Forecast head training (encoder frozen)')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Phase 2: unfreeze and fine-tune end-to-end ────────────────────────────
encoder.trainable = True

forecaster.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),   # 100× lower than Phase 1
    loss='mse'
)

early_stop_p2 = EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)
reduce_lr_p2 = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=7, min_lr=1e-8, verbose=1
)

history_p2 = forecaster.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop_p2, reduce_lr_p2],
    verbose=1
)

plt.figure(figsize=(10, 4))
plt.plot(history_p2.history['loss'],     label='Train loss')
plt.plot(history_p2.history['val_loss'], label='Val loss')
plt.title('Stage 2 Phase 2 — End-to-end fine-tuning')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Forecast the next 24 hours

In [ ]:
full_scaled   = np.concatenate([train_scaled, test_scaled], axis=0)
last_sequence = full_scaled[-time_steps:]              # (672, 4)
last_sequence = np.expand_dims(last_sequence, axis=0)  # (1, 672, 4)

scaled_forecast = forecaster.predict(last_sequence)    # (1, 96)

dummy = np.zeros((n_future, n_features))
dummy[:, target_col_idx] = scaled_forecast[0]
forecast_real = scaler.inverse_transform(dummy)[:, target_col_idx]

print(f"Forecast for the next {n_future} steps (sensor {target_sensor}):")
for i, val in enumerate(forecast_real, 1):
    print(f"  t+{i:3d}: {val:.2f}")

# Per-step evaluation on the test set

In [ ]:
test_preds_scaled = forecaster.predict(X_test)  # (samples, 96)

print(f"{'Step':<8} {'MAE':>8} {'RMSE':>8}")
print("-" * 26)

mae_per_step  = []
rmse_per_step = []

for step in range(n_future):
    dummy_p = np.zeros((len(test_preds_scaled), n_features))
    dummy_p[:, target_col_idx] = test_preds_scaled[:, step]
    preds_real = scaler.inverse_transform(dummy_p)[:, target_col_idx]

    dummy_t = np.zeros((len(y_test), n_features))
    dummy_t[:, target_col_idx] = y_test[:, step]
    true_real = scaler.inverse_transform(dummy_t)[:, target_col_idx]

    mae  = mean_absolute_error(true_real, preds_real)
    rmse = np.sqrt(mean_squared_error(true_real, preds_real))
    mae_per_step.append(mae)
    rmse_per_step.append(rmse)
    if step % 12 == 0:   # print every 3 hours to keep output manageable
        print(f"  t+{step+1:<4}  {mae:>7.2f}  {rmse:>7.2f}")

# ── Error-over-horizon plot ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
steps = range(1, n_future + 1)

axes[0].plot(steps, mae_per_step,  color='steelblue')
axes[0].set_title('MAE per forecast step')
axes[0].set_xlabel('Steps ahead (×15 min)')
axes[0].set_ylabel('MAE')
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, rmse_per_step, color='tomato')
axes[1].set_title('RMSE per forecast step')
axes[1].set_xlabel('Steps ahead (×15 min)')
axes[1].set_ylabel('RMSE')
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Sensor {target_sensor} — forecast error over 24-hour horizon', fontsize=13)
plt.tight_layout()
plt.show()

# Visual: predicted vs actual for the last test window

In [ ]:
last_pred_scaled = forecaster.predict(X_test[-1:])  # (1, 96)

dummy_p = np.zeros((n_future, n_features))
dummy_p[:, target_col_idx] = last_pred_scaled[0]
pred_real = scaler.inverse_transform(dummy_p)[:, target_col_idx]

dummy_t = np.zeros((n_future, n_features))
dummy_t[:, target_col_idx] = y_test[-1]
true_real = scaler.inverse_transform(dummy_t)[:, target_col_idx]

steps = range(1, n_future + 1)
plt.figure(figsize=(14, 5))
plt.plot(steps, true_real,  label='Actual',    color='steelblue', marker='o', markersize=3)
plt.plot(steps, pred_real,  label='Predicted', color='tomato',    marker='x', markersize=3, linestyle='--')
plt.title(f'Sensor {target_sensor}: 24-hour forecast vs actual (last test window)')
plt.xlabel('Steps ahead (×15 min)')
plt.ylabel('Conductivity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()